# Описание модели

В этом ноутбуке реализована модель для бинарной классификации изображений: `real / fake`.

Модель принимает RGB-изображение и предсказывает, относится ли оно к классу `fake`.  
Основная идея — извлекать визуальные признаки изображения с помощью сверточной нейронной сети и затем выполнять классификацию через финальный линейный слой.

---

## Архитектура модели

Используется сверточная RegNet-подобная архитектура с bottleneck-блоками, grouped convolution и Squeeze-and-Excitation.

Схема модели:

    Входное изображение
    [B, 3, 224, 224]
            |
            v
    Conv Stem
    Conv2d: 3 -> 32
    BatchNorm2d
    ReLU
            |
            v
    Stage 1
    ResBottleneckBlock x 2
    width = 232
            |
            v
    Stage 2
    ResBottleneckBlock x 5
    width = 696
            |
            v
    Stage 3
    ResBottleneckBlock x 12
    width = 1392
            |
            v
    Stage 4
    ResBottleneckBlock x 1
    width = 3712
            |
            v
    AdaptiveAvgPool2d(1, 1)
            |
            v
    Flatten
            |
            v
    Linear classifier
            |
            v
    Output logits
    [B, 2]

Основные элементы модели:

- сверточный stem для начального извлечения признаков;
- несколько последовательных stages с bottleneck-блоками;
- grouped convolution внутри блоков;
- Squeeze-and-Excitation для переоценки важности каналов;
- residual connections для стабильного обучения;
- global average pooling перед классификатором;
- финальный Linear-слой на 2 класса.

---

## Аугментации

Во время обучения используются аугментации, которые повышают устойчивость модели к изменениям изображения.

Train-аугментации:

    RandomResizedCrop 224x224
    HorizontalFlip
    ShiftScaleRotate
    RandomBrightnessContrast
    GaussNoise
    Sharpen
    Normalize ImageNet mean/std
    ToTensorV2

Validation/Test обработка:

    Resize 224x224
    Normalize ImageNet mean/std
    ToTensorV2

---

## Обучение

Для обучения используется `CrossEntropyLoss` с весами классов и label smoothing.

Оптимизатор:

    AdamW
    learning rate = 3e-4
    weight decay = 1e-4

Scheduler:

    OneCycleLR

Также используется gradient clipping для стабилизации обучения.

---

## Инференс

Модель возвращает logits для двух классов.

Вероятность класса `fake` считается так:

    probs = torch.softmax(logits, dim=1)[:, 1]

Для финального предсказания используется подобранный threshold.

Также применяется Test-Time Augmentation:

    1. Предсказание на исходном изображении
    2. Предсказание на горизонтально отражённом изображении
    3. Усреднение logits
    4. Softmax
    5. Применение threshold

Финальный submission сохраняется в формате:

    id,target_feature

# Загрузка библиотек

In [ ]:
import glob
import os
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from collections import OrderedDict
from typing import Optional, Sequence
import math

# Сиды и переход на cude (если возможно)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

In [ ]:
import warnings

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Классы обработки входный данных

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

class TrainDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]["id"]
        label = self.df.iloc[idx]["label"]

        img_path = os.path.join(self.root_dir, f"{img_id}.jpg")
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = np.array(image)
            image = self.transform(image=image)["image"]

        return image, label

In [ ]:
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.files = os.listdir(root_dir)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_name = self.files[idx]
        path = os.path.join(self.root_dir, file_name)

        image = Image.open(path).convert("RGB")

        if self.transform:
            image = np.array(image)
            image = self.transform(image=image)["image"]

        return image, file_name

# Дата трансформеры

In [ ]:
!pip install -U albumentations==1.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: albumentations
    Found existing installation: albumentations 2.0.8
    Uninstalling albumentations-2.0.8:
      Successfully uninstalled albumentations-2.0.8


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transforms = A.Compose([
    A.RandomResizedCrop(224, 224, scale=(0.85, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=10,
        p=0.5
    ),

    A.RandomBrightnessContrast(p=0.3),

    A.GaussNoise(p=0.2),

    A.Sharpen(p=0.1),

    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(height=224, width=224, p=1.0),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2(),
])

test_transforms = val_transforms

# Загрузка данных

In [ ]:
import pandas as pd

In [ ]:
labels_for_train = pd.read_csv("/content/drive/MyDrive/YAN/train_solution.csv", header=None)

In [ ]:
labels_for_train.columns = ["id", "label"]

In [ ]:
labels_for_train

,id,label
0,0,0
1,1,1
2,2,1
3,3,0
4,4,0
...,...,...
49995,49995,0
49996,49996,0
49997,49997,0
49998,49998,0


In [ ]:
labels_for_train.value_counts("label")

,count
label,
0,41500
1,8500


In [ ]:
train_df, val_df = train_test_split(
    labels_for_train,
    test_size=0.2,
    stratify=labels_for_train["label"],
    random_state=42
)

In [ ]:
!unzip -q /content/drive/MyDrive/YAN/train_images.zip -d /content/

In [ ]:
!unzip -q /content/drive/MyDrive/YAN/test_images.zip -d /content/

In [ ]:
print(os.listdir("/content"))

['.config', 'test_images', 'train_images', 'drive', 'sample_data']


In [ ]:
print(len(os.listdir("/content/train_images")))

50000


In [ ]:
from torch.utils.data import DataLoader

train_dataset = TrainDataset(train_df, "/content/train_images", transform=train_transforms)
val_dataset   = TrainDataset(val_df,   "/content/train_images", transform=test_transforms)
test_dataset  = TestDataset("/content/test_images", transform=test_transforms)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)

In [ ]:
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)

# Архитектура модели основанная на семействе RegNet

In [ ]:
from collections import OrderedDict
from typing import Optional, Sequence

import torch
from torch import nn


class Conv2dNormActivation(nn.Sequential):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: Optional[int] = None,
        groups: int = 1,
        bias: bool = False,
        activation_layer: Optional[type[nn.Module]] = nn.ReLU,
    ) -> None:
        if padding is None:
            padding = kernel_size // 2

        layers = [
            ("0", nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                groups=groups,
                bias=bias,
            )),
            ("1", nn.BatchNorm2d(
                out_channels,
                eps=1e-5,
                momentum=0.1,
                affine=True,
                track_running_stats=True,
            )),
        ]
        if activation_layer is not None:
            layers.append(("2", activation_layer(inplace=True)))
        super().__init__(OrderedDict(layers))


class SqueezeExcitation(nn.Module):
    def __init__(self, input_channels: int, squeeze_channels: int) -> None:
        super().__init__()
        self.avgpool = nn.AdaptiveAvgPool2d(output_size=1)
        self.fc1 = nn.Conv2d(input_channels, squeeze_channels, kernel_size=1, stride=1)
        self.fc2 = nn.Conv2d(squeeze_channels, input_channels, kernel_size=1, stride=1)
        self.activation = nn.ReLU()
        self.scale_activation = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.avgpool(x)
        scale = self.fc1(scale)
        scale = self.activation(scale)
        scale = self.fc2(scale)
        scale = self.scale_activation(scale)
        return x * scale


class BottleneckTransform(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int,
        groups: int,
        se_squeeze_channels: int,
    ) -> None:
        super().__init__()
        self.a = Conv2dNormActivation(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            groups=1,
            bias=False,
            activation_layer=nn.ReLU,
        )
        self.b = Conv2dNormActivation(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=groups,
            bias=False,
            activation_layer=nn.ReLU,
        )
        self.se = SqueezeExcitation(out_channels, se_squeeze_channels)
        self.c = Conv2dNormActivation(
            out_channels,
            out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            groups=1,
            bias=False,
            activation_layer=None,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.a(x)
        x = self.b(x)
        x = self.se(x)
        x = self.c(x)
        return x


class ResBottleneckBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int,
        groups: int,
        se_squeeze_channels: int,
    ) -> None:
        super().__init__()
        self.proj = None
        if stride != 1 or in_channels != out_channels:
            self.proj = Conv2dNormActivation(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                padding=0,
                groups=1,
                bias=False,
                activation_layer=None,
            )
        self.f = BottleneckTransform(
            in_channels=in_channels,
            out_channels=out_channels,
            stride=stride,
            groups=groups,
            se_squeeze_channels=se_squeeze_channels,
        )
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x if self.proj is None else self.proj(x)
        out = self.f(x)
        out = out + identity
        out = self.activation(out)
        return out


class AnyStage(nn.Sequential):
    pass


class SimpleStemIN(nn.Sequential):
    def __init__(self) -> None:
        super().__init__(OrderedDict([
            ("0", nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False)),
            ("1", nn.BatchNorm2d(32, eps=1e-5, momentum=0.1, affine=True, track_running_stats=True)),
            ("2", nn.ReLU(inplace=True)),
        ]))


class RegNet(nn.Module):
    def __init__(
        self,
        depths: Sequence[int] = (2, 5, 12, 1),
        widths: Sequence[int] = (232, 696, 1392, 3712),
        group_width: int = 232,
        num_classes: int = 1000,
    ) -> None:
        super().__init__()
        assert len(depths) == 4 and len(widths) == 4

        self.stem = SimpleStemIN()

        stage_dict = OrderedDict()
        in_channels = 32

        for stage_idx, (depth, out_channels) in enumerate(zip(depths, widths), start=1):
            blocks = OrderedDict()
            for block_idx in range(depth):
                stride = 2 if block_idx == 0 else 1
                block_in = in_channels if block_idx == 0 else out_channels
                groups = out_channels // group_width
                se_squeeze_channels = block_in // 4

                blocks[f"block{stage_idx}-{block_idx}"] = ResBottleneckBlock(
                    in_channels=block_in,
                    out_channels=out_channels,
                    stride=stride,
                    groups=groups,
                    se_squeeze_channels=se_squeeze_channels,
                )

            stage_dict[f"block{stage_idx}"] = AnyStage(blocks)
            in_channels = out_channels

        self.trunk_output = nn.Sequential(stage_dict)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(widths[-1], num_classes, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.trunk_output(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


def regnet_y_16gf(num_classes: int = 1000) -> RegNet:
    return RegNet(
        depths=(2, 5, 12, 1),
        widths=(232, 696, 1392, 3712),
        group_width=232,
        num_classes=num_classes,
    )


# Функции для обучения, валидации, вывода графика, подсчета метрик

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sns.set(style='darkgrid')


def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    train_accuracy: list[float],
    valid_accuracy: list[float],
    title: str
):
    plt.figure(figsize=(16, 8))

    plt.title(title + ' loss')

    plt.plot(train_loss, label='Train loss')
    plt.plot(valid_loss, label='Valid loss')
    plt.legend()

    plt.show()

    plt.figure(figsize=(16, 8))

    plt.title(title + ' accuracy')

    plt.plot(train_accuracy, label='Train accuracy')
    plt.plot(valid_accuracy, label='Valid accuracy')
    plt.legend()

    plt.show()

In [ ]:
def train(model: nn.Module, data_loader: DataLoader, optimizer, loss_fn, scheduler=None):
    model.train()

    total_loss = 0
    total_correct = 0

    for x, y in tqdm(data_loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        output = model(x)
        loss = loss_fn(output, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        total_correct += (output.argmax(dim=1) == y).sum().item()

    return total_loss / len(data_loader), total_correct / len(data_loader.dataset)

In [ ]:
@torch.inference_mode()
def evaluate(model: nn.Module, data_loader: DataLoader, loss_fn):
    model.eval()

    total_loss = 0
    total_correct = 0

    for x, y in tqdm(data_loader):
        x, y = x.to(device), y.to(device)
        output = model(x)

        loss = loss_fn(output, y)

        total_loss += loss.item()

        total_correct += (output.argmax(dim=1) == y).sum().item()

    return total_loss / len(data_loader), total_correct / len(data_loader.dataset)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def get_metrics(model, dataloader):
    model.eval()

    all_preds = []
    all_true = []

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device)

        pred = model(X)

        predicted_classes = torch.argmax(pred, dim=1)

        all_preds.extend(predicted_classes.cpu().numpy())
        all_true.extend(y.cpu().numpy())

    acc = accuracy_score(all_true, all_preds)
    prec = precision_score(all_true, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_true, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_true, all_preds, average='macro', zero_division=0)

    return acc, prec, rec, f1

In [ ]:
from IPython.display import clear_output
import os
import torch


def fit(model, train_loader, valid_loader, test_loader, optimizer, loss_fn, scheduler, num_epochs, title):
    train_loss_history, valid_loss_history = [], []
    min_valid_error = float('+inf')
    train_accuracy_history, valid_accuracy_history = [], []

    save_dir = "/content/drive/MyDrive/YAN"
    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(num_epochs):
        train_loss, train_accuracy = train(model, train_loader, optimizer, loss_fn, scheduler)
        valid_loss, valid_accuracy = evaluate(model, valid_loader, loss_fn)
        min_valid_error = min(min_valid_error, valid_loss)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        train_accuracy_history.append(train_accuracy)
        valid_accuracy_history.append(valid_accuracy)

        save_path = os.path.join(
            save_dir,
            f"{epoch+1}_{valid_accuracy:.4f}.pth"
        )
        torch.save(model.state_dict(), save_path)

        clear_output()

        plot_stats(
            train_loss_history, valid_loss_history,
            train_accuracy_history, valid_accuracy_history,
            title
        )

In [ ]:
!ls /content/drive/MyDrive/

'Colab Notebooks'   YAN


# Загрузка модели

In [ ]:
model = regnet_y_16gf()
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model.to(device)

RegNet(
  (stem): SimpleStemIN(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (trunk_output): Sequential(
    (block1): AnyStage(
      (block1-0): ResBottleneckBlock(
        (proj): Conv2dNormActivation(
          (0): Conv2d(32, 232, kernel_size=(1, 1), stride=(2, 2), bias=False)
          (1): BatchNorm2d(232, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (f): BottleneckTransform(
          (a): Conv2dNormActivation(
            (0): Conv2d(32, 232, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(232, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (b): Conv2dNormActivation(
            (0): Conv2d(232, 232, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
            (1):

# Настройка оптимизатора + выбор лосс функции

In [ ]:
class_counts = train_df["label"].value_counts().sort_index()

In [ ]:
class_counts

,count
label,
0,33200
1,6800


In [ ]:
counts = torch.tensor([33200, 6800], dtype=torch.float32, device=device)
weights = counts.sum() / (len(counts) * counts)

num_epochs = 50

criterion = nn.CrossEntropyLoss(
    weight=weights,
    label_smoothing=0.05,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-4,
    epochs=num_epochs,
    steps_per_epoch=len(train_loader)
)

# Обучение модели

In [ ]:
fit(
    model,
    train_loader,
    val_loader,
    test_loader,
    optimizer,
    criterion,
    scheduler,
    num_epochs,
    title="RegNet training"
)

NameError: name 'fit' is not defined

In [ ]:
acc, prec, rec, f1 = get_metrics(model, val_loader)

In [ ]:
acc, prec, rec, f1

(0.6433, 0.5532126734999578, 0.5865591778880227, 0.5404395127246784)

# Модель 17_0.9085.pth

In [ ]:
model_17 = regnet_y_16gf(num_classes=2)
model_17.to(device)
model_17.load_state_dict(
    torch.load("/content/drive/MyDrive/YAN/17_0.9085.pth", map_location=device)
)


<All keys matched successfully>

acc, prec, rec, f1

In [ ]:
get_metrics(model=model_17, dataloader=val_loader)

(0.9085, 0.8285322452612969, 0.8751842664776754, 0.8488828204193299)

# Модель 14_0.8902.pth

In [ ]:
model_14 = regnet_y_16gf(num_classes=2)
model_14.to(device)
model_14.load_state_dict(
    torch.load("/content/drive/MyDrive/YAN/14_0.8902.pth", map_location=device)
)

<All keys matched successfully>

acc, prec, rec, f1

In [ ]:
get_metrics(model=model_14, dataloader=val_loader)

(0.8902, 0.8014944769330734, 0.8288447909284196, 0.8140243902439024)

# Модель 12_0.8591.pth

In [ ]:
model_12 = regnet_y_16gf(num_classes=2)
model_12.to(device)
model_12.load_state_dict(
    torch.load("/content/drive/MyDrive/YAN/12_0.8591.pth", map_location=device)
)

<All keys matched successfully>

acc, prec, rec, f1

In [ ]:
get_metrics(model=model_12, dataloader=val_loader)

(0.8591, 0.7582567693600124, 0.8405138199858257, 0.7865887428303322)

# Модель 16_0.8669.pth

In [ ]:
model_16 = regnet_y_16gf(num_classes=2)
model_16.to(device)
model_16.load_state_dict(
    torch.load("/content/drive/MyDrive/YAN/16_0.8669.pth", map_location=device)
)

<All keys matched successfully>

acc, prec, rec, f1

In [ ]:
get_metrics(model=model_16, dataloader=val_loader)

(0.8669, 0.7707212494408346, 0.8681325301204819, 0.802476769168164)

# Модель 9_0.8492.pth

In [ ]:
model_9 = regnet_y_16gf(num_classes=2)
model_9.to(device)
model_9.load_state_dict(
    torch.load("/content/drive/MyDrive/YAN/9_0.8492.pth", map_location=device)
)

<All keys matched successfully>

acc, prec, rec, f1

In [ ]:
get_metrics(model=model_9, dataloader=val_loader)

(0.8492, 0.73936707251406, 0.7903472714386959, 0.7594602313112084)

# Предикт №1

In [ ]:
model = model_17

In [ ]:
all_preds = []
all_ids = []

for X, file_names in test_loader:
    X = X.to(device)

    with torch.no_grad():
        logits = model(X)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

    all_preds.extend(preds.cpu().numpy())

    for name in file_names:
        id_ = int(name.split(".")[0])  # "123.jpg" → 123
        all_ids.append(id_)

In [ ]:
df = pd.DataFrame({
    "id": all_ids,
    "target_feature": all_preds
})

In [ ]:
df = df.sort_values("id").reset_index(drop=True)

In [ ]:
df.to_csv("/content/drive/MyDrive/YAN/submission.csv", index=False)

# Тюнинг - подбор пробы

In [ ]:
model.eval()

all_probs = []
all_true = []

with torch.no_grad():
    for X, y in val_loader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        probs = torch.softmax(logits, dim=1)[:, 1]

        all_probs.extend(probs.cpu().numpy())
        all_true.extend(y.cpu().numpy())

all_probs = np.array(all_probs)
all_true = np.array(all_true)

In [ ]:
best_f1 = 0
best_threshold = 0

for t in np.arange(0.1, 0.9, 0.01):
    preds = (all_probs > t).astype(int)
    f1 = f1_score(all_true, preds, average='macro')

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

Best threshold: 0.7199999999999996
Best F1: 0.863317365846364


# Предикт №2

In [ ]:
threshold = 0.72

all_preds = []
all_ids = []

for X, file_names in test_loader:
    X = X.to(device)

    with torch.no_grad():
        logits = model(X)

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = (probs > threshold).long()

    all_preds.extend(preds.cpu().numpy())

    for name in file_names:
        id_ = int(name.split(".")[0])
        all_ids.append(id_)

In [ ]:
df = pd.DataFrame({
    "id": all_ids,
    "target_feature": all_preds
})

In [ ]:
df = df.sort_values("id").reset_index(drop=True)

In [ ]:
df.to_csv("/content/drive/MyDrive/YAN/submission_try2.csv", index=False)

# Предикт № 3 + тюнинг TTA

In [ ]:
threshold = 0.72

all_preds = []
all_ids = []

for X, file_names in test_loader:
    X = X.to(device)

    with torch.no_grad():
        logits1 = model(X)

        X_flip = torch.flip(X, dims=[3]).contiguous()
        logits2 = model(X_flip)

        logits = (logits1 + logits2) / 2

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = (probs > threshold).long()

    all_preds.extend(preds.cpu().numpy())

    for name in file_names:
        id_ = int(name.split(".")[0])
        all_ids.append(id_)

In [ ]:
df = pd.DataFrame({
    "id": all_ids,
    "target_feature": all_preds
})

In [ ]:
df.to_csv("/content/drive/MyDrive/YAN/submission_try3.csv", index=False)

# Тюнинг поиск threshold с TTA

In [ ]:
model.eval()

all_probs = []
all_true = []

with torch.no_grad():
    for X, y in val_loader:
        X = X.to(device)
        y = y.to(device)

        logits1 = model(X)

        X_flip = torch.flip(X, dims=[3]).contiguous()
        logits2 = model(X_flip)

        logits = (logits1 + logits2) / 2

        probs = torch.softmax(logits, dim=1)[:, 1]

        all_probs.extend(probs.cpu().numpy())
        all_true.extend(y.cpu().numpy())

all_probs = np.array(all_probs)
all_true = np.array(all_true)

In [ ]:
best_f1 = 0
best_threshold = 0

for t in np.arange(0.1, 0.9, 0.01):
    preds = (all_probs > t).astype(int)
    f1 = f1_score(all_true, preds, average='macro')

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best threshold (TTA):", best_threshold)
print("Best F1 (TTA):", best_f1)

Best threshold (TTA): 0.6599999999999997
Best F1 (TTA): 0.8693929330997816


# Предикт №4 новый threshold расчитанный на TTA

In [ ]:
threshold = 0.66
all_preds = []
all_ids = []

for X, file_names in test_loader:
    X = X.to(device)

    with torch.no_grad():
        logits1 = model(X)

        X_flip = torch.flip(X, dims=[3]).contiguous()
        logits2 = model(X_flip)

        logits = (logits1 + logits2) / 2

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = (probs > threshold).long()

    all_preds.extend(preds.cpu().numpy())

    for name in file_names:
        id_ = int(name.split(".")[0])
        all_ids.append(id_)

In [ ]:
df = pd.DataFrame({
    "id": all_ids,
    "target_feature": all_preds
})

In [ ]:
df.to_csv("/content/drive/MyDrive/YAN/submission_try4.csv", index=False)